# VoiceGuard: IBM LinuxONE (s390x) Real-Time Voice Clone Detection

**IBM Z Datathon 2026** — *Track: AI Secured: Innovation Without Exposure*

This interactive notebook demonstrates the **pure-NumPy LCNN acoustic classifier** running locally on an IBM LinuxONE / s390x environment without requiring `onnxruntime` or GPU acceleration.

### Pipeline Overview:
1. **Audio Ingestion**: 16 kHz mono PCM audio (1.0 s sliding window, 250 ms stride).
2. **DSP Feature Extraction**: 20 Linear Frequency Cepstral Coefficients (LFCC) across 40 linear filterbank bins.
3. **Model Inference**: Pure-NumPy Max-Feature-Map CNN (LCNN) loading per-tensor float32 weights.
4. **Visualization**: Audio waveform, LFCC heatmap, and sliding risk progression with mitigation thresholds.

In [ ]:
import sys
import os

# Add repository directories to path
REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(REPO_ROOT, "voiceguard"))
sys.path.insert(0, os.path.join(REPO_ROOT, "gateway"))

import numpy as np
import matplotlib.pyplot as plt
from numpy_lcnn import NumpyLCNN
from dsp_pipeline import compute_lfcc, SAMPLE_RATE, FRAME_SAMPLES

print(f"NumPy version: {np.__version__}")
print(f"Target sample rate: {SAMPLE_RATE} Hz")

## 1. Load the Pure-NumPy LCNN Model
On IBM LinuxONE (s390x, big-endian), no official `onnxruntime` wheel exists. `NumpyLCNN` performs the exact convolution, MFM, and batch-norm operations using vectorized NumPy operations, achieving ~1.3 ms inference per 1-second window.

In [ ]:
WEIGHTS_DIR = os.path.join(REPO_ROOT, "voiceguard", "model", "track_a_npz")
model = NumpyLCNN(WEIGHTS_DIR)

# Smoke test with golden zeros and ramp inputs
zeros_in = np.zeros((1, 98, 20), dtype=np.float32)
zeros_logits = model.logits(zeros_in)
print(f"Zeros logits: {zeros_logits.squeeze()} (Expected ~ [-28.1055, 19.7470])")

ramp_in = (np.arange(98 * 20, dtype=np.float32).reshape(1, 98, 20) / (98 * 20) - 0.5)
ramp_logits = model.logits(ramp_in)
print(f"Ramp logits:  {ramp_logits.squeeze()} (Expected ~ [-19.9592, 14.1075])")

## 2. Synthesize Speech-Like Test Audio
We generate a 5.0-second speech-like harmonic signal with natural pitch inflection, pauses, and low background noise.

In [ ]:
duration_s = 5.0
t = np.arange(int(SAMPLE_RATE * duration_s)) / SAMPLE_RATE
f0 = 130 + 30 * np.sin(2 * np.pi * 0.8 * t)
phase = 2 * np.pi * np.cumsum(f0) / SAMPLE_RATE
harmonics = sum(np.sin(k * phase) / (k ** 0.8) for k in range(1, 30) if k * 140 < SAMPLE_RATE / 2)
envelope = np.convolve((np.sin(2 * np.pi * 1.5 * t) > -0.2).astype(float), np.ones(400)/400, mode='same')
noise = np.random.default_rng(42).normal(0, 0.005, len(t))
audio = (0.4 * harmonics / np.max(np.abs(harmonics)) * envelope + noise).astype(np.float32)

print(f"Generated audio: {len(audio)} samples ({duration_s:.1f} s)")

## 3. Sliding Window Inference
Stride across audio in 250 ms steps (4,000 samples) with a 1.0 s window (16,000 samples) to evaluate real-time scoring.

In [ ]:
stride_samples = 4000
window_samples = 16000

timestamps = []
synthetic_scores = []
lfcc_frames = []

for start in range(0, len(audio) - window_samples + 1, stride_samples):
    window = audio[start : start + window_samples]
    lfcc = compute_lfcc(window)  # Shape: (98, 20)
    batch = lfcc[np.newaxis, :, :]
    
    # Forward pass
    logits = model.logits(batch).squeeze()
    
    # Softmax over logits [bonafide, spoof]
    exp_l = np.exp(logits - np.max(logits))
    p_synthetic = float(exp_l[1] / np.sum(exp_l))
    
    t_center = (start + window_samples / 2) / SAMPLE_RATE
    timestamps.append(t_center)
    synthetic_scores.append(p_synthetic)
    lfcc_frames.append(lfcc)

print(f"Evaluated {len(synthetic_scores)} sliding windows.")
print(f"Mean P(synthetic): {np.mean(synthetic_scores):.4f}")

## 4. Visualizing Features and Risk Trajectory

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(11, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1.2, 1.2]})

# 1. Audio Waveform
axs[0].plot(t, audio, color='#3b82f6', lw=0.8)
axs[0].set_ylabel('Amplitude')
axs[0].set_title('Inbound Audio Stream (16 kHz PCM16)', fontweight='bold')
axs[0].grid(True, alpha=0.2)

# 2. LFCC Feature Spectrogram (Middle Window)
mid_idx = len(lfcc_frames) // 2
im = axs[1].imshow(lfcc_frames[mid_idx].T, aspect='auto', origin='lower', cmap='viridis',
                   extent=[timestamps[mid_idx] - 0.5, timestamps[mid_idx] + 0.5, 0, 20])
axs[1].set_ylabel('LFCC Bin (0-19)')
axs[1].set_title('LFCC Acoustic Heatmap (1.0 s Window)', fontweight='bold')

# 3. Risk Progression and Mitigation Levels
axs[2].plot(timestamps, synthetic_scores, marker='o', color='#ef4444', lw=2, label='P(synthetic)')
axs[2].axhline(0.35, color='#22c55e', linestyle='--', alpha=0.7, label='ALLOW Threshold (0.35)')
axs[2].axhline(0.70, color='#f59e0b', linestyle='--', alpha=0.7, label='CHALLENGE Threshold (0.70)')
axs[2].axhline(0.85, color='#ef4444', linestyle=':', alpha=0.7, label='BLOCK / SEVER (0.85)')
axs[2].set_xlabel('Time (seconds)')
axs[2].set_ylabel('Risk Score')
axs[2].set_ylim(-0.05, 1.05)
axs[2].set_title('Temporal Risk Trajectory & Decision Gate', fontweight='bold')
axs[2].legend(loc='lower right', framealpha=0.8)
axs[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()